# Bagging Cuts Variance, Not Bias

**Topic:** Bagging &nbsp;·&nbsp; **Difficulty:** Hard

"Ensembling makes things better" is true but useless. The precise claim is
sharper: for a high-variance base learner the squared bias is a rounding error
next to the variance, and averaging bootstrap-trained models roughly halves the
variance while leaving bias negligible, with the floor set by how correlated
the models are.

Measure both halves separately, on 60 independently drawn training sets, and
watch which one moves.

## The data (created in the next cell)

- A fixed test grid `X_test` `(200, 1)` and the true function `f_true`.
- 60 independent training sets of 120 points each: for training set `i`,
  `r = np.random.default_rng(1000 + i)`, `Xtr = r.uniform(-2, 2, (120, 1))` and
  `ytr = f_true_fn(Xtr.ravel()) + r.normal(0, 0.3, 120)`. Use that same `r` for
  the bootstrap draws.
- Base learner: `DecisionTreeRegressor(max_depth=None, random_state=0)`, high
  variance by design.

## Your task

With **sklearn and NumPy**, for each of the 60 training sets:

1. Fit one deep tree and predict `X_test`, collecting into `single_preds`
   `(60, 200)`.
2. Fit a bag of 25 trees, each on a bootstrap resample of that same training set
   (use `np.random.default_rng(1000 + i)` for training set `i`), average their
   predictions, collecting into `bag_preds` `(60, 200)`.
3. For each method compute, averaged over the test grid:
   - `bias2` = mean over test points of `(mean prediction - f_true)^2`,
   - `variance` = mean over test points of the prediction variance across the 60
     runs (use `np.var` with default `ddof=0`).
4. Set `answers` to a dict with `"bias2_single"`, `"var_single"`,
   `"bias2_bag"`, `"var_bag"`, `"var_ratio"` = `var_bag / var_single`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks variance fell by at least 40% and that squared bias stayed under 5% of
the single model's variance.

In [ ]:
import numpy as np

def f_true_fn(x):
    return np.sin(2.5 * x) + 0.4 * x ** 2

X_test = np.linspace(-2, 2, 200).reshape(-1, 1)
f_true = f_true_fn(X_test.ravel())
n_train, n_runs, n_bag = 120, 60, 25
print(f"{n_runs} training sets of {n_train} points, bags of {n_bag} trees")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("bias2_single", [], 0.0013758410137895236, -0.0002303680880492858, 1e-06),
    ("var_single", [], 0.09401741076151227, -0.015742088615906945, 1e-06),
    ("bias2_bag", [], 0.0008089730750808543, -0.00013545284572991903, 1e-06),
    ("var_bag", [], 0.045940890496266394, -0.007692251503505136, 1e-06),
    ("var_ratio", [], 0.48864237085619816, -0.0818173085304121, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "var_ratio" in answers:
    _v = answers["var_ratio"] < 0.6
    _b = answers["bias2_bag"] < 0.05 * answers["var_single"]
    print(f"  {'variance falls':<24} {'ok' if _v else 'wrong'}  (ratio = {answers['var_ratio']:.3f})")
    print(f"  {'bias stays negligible':<24} {'ok' if _b else 'wrong'}  "
          f"(bias2 {answers['bias2_bag']:.5f} vs var {answers['var_single']:.5f})")
    _ok = _ok and _v and _b

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Each bootstrap tree sees a slightly different sample, so their errors are only
partly correlated, and averaging cancels the independent part. The residual variance
is the correlated part, which is why random forests go further and decorrelate
the trees by sampling features too. Squared bias stays around 1% of the variance throughout: averaging models that
target the same function does not make them more accurate on average, only more
stable, which is why bagging helps deep trees and does nothing for a linear
model.

```python
import numpy as np
from sklearn.tree import DecisionTreeRegressor

single_preds = np.zeros((n_runs, len(X_test)))
bag_preds = np.zeros((n_runs, len(X_test)))

for i in range(n_runs):
    r = np.random.default_rng(1000 + i)
    Xtr = r.uniform(-2, 2, (n_train, 1))
    ytr = f_true_fn(Xtr.ravel()) + r.normal(0, 0.3, n_train)

    single_preds[i] = DecisionTreeRegressor(random_state=0).fit(Xtr, ytr).predict(X_test)

    acc = np.zeros(len(X_test))
    for _ in range(n_bag):
        idx = r.integers(0, n_train, n_train)
        acc += DecisionTreeRegressor(random_state=0).fit(Xtr[idx], ytr[idx]).predict(X_test)
    bag_preds[i] = acc / n_bag

def decompose(P):
    return (float(np.mean((P.mean(axis=0) - f_true) ** 2)),
            float(np.mean(P.var(axis=0))))

bias2_single, var_single = decompose(single_preds)
bias2_bag, var_bag = decompose(bag_preds)

answers = {
    "bias2_single": bias2_single,
    "var_single": var_single,
    "bias2_bag": bias2_bag,
    "var_bag": var_bag,
    "var_ratio": var_bag / var_single,
}
```